# EHALC–Foxtsage: Revision-Ready Experimental Notebook

This notebook is a cleaned, reviewer-oriented replacement for the previous experimental notebook.

## Design goals

1. **One auditable data pipeline** — no duplicate train/test pipelines.
2. **No preprocessing leakage** — scaling is fitted only on the fitting subset.
3. **Fixed fit / validation / test partitions** — 64% / 16% / 20% with stratification.
4. **Untouched final test set** — the test labels are never used for model selection.
5. **Component-level ablation** — original ALC+IFOX, original ALC+adapted Foxtsage search, enhanced ALC+IFOX, and full enhanced framework.
6. **Repeated-seed evaluation** — quantify optimizer variability.
7. **Single source of truth for final metrics** — all headline metrics, ROC/PR curves and the confusion matrix are generated from the same saved prediction file.
8. **Global explainability** — aggregate SHAP feature importance rather than a single representative sample.
9. **Reproducibility artifacts** — partitions, configuration, optimization history, predictions and result tables are saved to disk.

### Important terminology note

The published Foxtsage work describes Foxtsage as a **FOX-TSA mechanism for dynamically selecting the SGD learning rate**.  
The direct population-based weight search used in the previous manuscript notebook is therefore treated here as an **ALC-adapted / enhanced Foxtsage-inspired search**, not silently presented as an exact reproduction of the published Foxtsage optimizer.

Before final resubmission, the manuscript should describe this adaptation explicitly.

In [ ]:
# Optional Colab install
# !pip install -q numpy pandas scikit-learn scipy matplotlib shap

from pathlib import Path
import json
import time
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DATA_PATH = Path("EVSE-B-HPC-Kernel-Events-Combined.csv")
OUTPUT_DIR = Path("revision_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Use smaller budgets while debugging. Set False for the paper run.
QUICK_MODE = True

# Paper-grade recommendation: 5-10 stochastic seeds.
SEEDS = [42, 52] if QUICK_MODE else [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]

# Optimizer budget
IFOX_ITERATIONS = 8 if QUICK_MODE else 120
IFOX_POPULATION = 8 if QUICK_MODE else 60
FOXT_ITERATIONS = 8 if QUICK_MODE else 120
FOXT_POPULATION = 8 if QUICK_MODE else 60
LOBULES_GRID = (8, 16) if QUICK_MODE else (16, 32, 64)

# Explainability budget
RUN_SHAP = False if QUICK_MODE else True
SHAP_BACKGROUND_N = 30
SHAP_EXPLAIN_N = 50

print({
    "quick_mode": QUICK_MODE,
    "seeds": SEEDS,
    "data_path": str(DATA_PATH),
    "output_dir": str(OUTPUT_DIR),
})

## 1. Load and validate the EVSE-B-HPC data

The earlier notebook mapped every non-`attack` label to benign. The source CSV contains malformed/header-like rows, so this revision **keeps only valid labels** before mapping the binary target.

Metadata columns such as `Attack`, `Scenario`, `State`, `interface`, and `time` are retained separately for audit/cross-scenario analysis but are **never used as predictor features**.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

metadata_candidates = ["time", "Attack", "Scenario", "State", "interface"]
available_metadata = [c for c in metadata_candidates if c in df.columns]

# Normalize label and keep only valid binary rows.
label_norm = df["Label"].astype(str).str.strip().str.lower()
valid_mask = label_norm.isin(["attack", "benign", "0"])

invalid_rows = df.loc[~valid_mask, available_metadata + ["Label"]].copy()
invalid_rows.to_csv(OUTPUT_DIR / "excluded_invalid_rows.csv", index=False)

work = df.loc[valid_mask].copy()
work["target"] = (work["Label"].astype(str).str.strip().str.lower() == "attack").astype(int)

metadata = work[available_metadata + ["target"]].copy()

drop_cols = set(available_metadata + ["Label", "target"])
drop_cols.update([c for c in work.columns if "unnamed" in c.lower()])

feature_df = work.drop(columns=[c for c in drop_cols if c in work.columns])
feature_df = feature_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

X_all = feature_df.to_numpy(dtype=np.float64)
y_all = work["target"].to_numpy(dtype=np.int64)
feature_names = feature_df.columns.to_list()

print("Raw rows:", len(df))
print("Valid rows:", len(work))
print("Excluded invalid rows:", len(invalid_rows))
print("Predictor features:", X_all.shape[1])
print("Class counts:", dict(zip(*np.unique(y_all, return_counts=True))))

pd.Series(feature_names, name="feature").to_csv(
    OUTPUT_DIR / "retained_features.csv", index=False
)

## 2. Freeze one stratified 64/16/20 split

The split is created from row indices first and saved. This makes every model use exactly the same observations.

- **Fit:** 64% of all valid observations
- **Validation:** 16%
- **Final test:** 20%

The test set is not used for hyperparameter selection, architecture selection, stopping, or optimizer fitness.

In [ ]:
all_idx = np.arange(len(y_all))

dev_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_all,
)

fit_idx, val_idx = train_test_split(
    dev_idx,
    test_size=0.20,  # 20% of 80% = 16% overall
    random_state=RANDOM_SEED,
    stratify=y_all[dev_idx],
)

np.savez(
    OUTPUT_DIR / "split_indices.npz",
    fit_idx=fit_idx,
    val_idx=val_idx,
    test_idx=test_idx,
)

def split_summary(name, idx):
    counts = np.bincount(y_all[idx], minlength=2)
    return {
        "split": name,
        "n": len(idx),
        "benign": int(counts[0]),
        "attack": int(counts[1]),
        "attack_rate": float(counts[1] / len(idx)),
    }

split_df = pd.DataFrame([
    split_summary("fit", fit_idx),
    split_summary("validation", val_idx),
    split_summary("test", test_idx),
])

display(split_df)
split_df.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

## 3. Leakage-safe standardization

The scaler is fitted **only on the fit subset** and then applied unchanged to validation and final test data.

This avoids the leakage present in the previous notebook, where `StandardScaler.fit_transform(X)` was called before the holdout split.

In [ ]:
scaler = StandardScaler()

X_fit = scaler.fit_transform(X_all[fit_idx])
X_val = scaler.transform(X_all[val_idx])
X_test = scaler.transform(X_all[test_idx])

y_fit = y_all[fit_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

def one_hot(y, n_classes=2):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out

y_fit_oh = one_hot(y_fit)
y_val_oh = one_hot(y_val)

print(X_fit.shape, X_val.shape, X_test.shape)

## 4. Shared evaluation utilities

All final tables and figures use the same helper functions.  
No metric is manually typed into the manuscript.

In [ ]:
def classification_metrics(y_true, y_pred, y_score):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "precision_attack": float(
            precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "recall_attack": float(
            recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "f1_attack": float(
            f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
    }

def save_predictions(path, row_indices, y_true, prob):
    pred = np.argmax(prob, axis=1)
    out = pd.DataFrame({
        "row_index": row_indices,
        "y_true": y_true,
        "prob_benign": prob[:, 0],
        "prob_attack": prob[:, 1],
        "y_pred": pred,
    })
    out.to_csv(path, index=False)
    return out

def evaluate_prediction_file(path):
    p = pd.read_csv(path)
    m = classification_metrics(
        p["y_true"].to_numpy(),
        p["y_pred"].to_numpy(),
        p["prob_attack"].to_numpy(),
    )
    cm = confusion_matrix(p["y_true"], p["y_pred"])
    return m, cm

def bootstrap_ci(y_true, y_pred, y_score, metric="accuracy", n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        ys = y_score[idx]

        # ROC-AUC/PR-AUC require both classes in a bootstrap sample.
        if metric in {"roc_auc", "pr_auc"} and len(np.unique(yt)) < 2:
            continue

        if metric == "accuracy":
            v = accuracy_score(yt, yp)
        elif metric == "f1_weighted":
            v = f1_score(yt, yp, average="weighted", zero_division=0)
        elif metric == "roc_auc":
            v = roc_auc_score(yt, ys)
        elif metric == "pr_auc":
            v = average_precision_score(yt, ys)
        else:
            raise ValueError(metric)
        values.append(v)

    lo, hi = np.percentile(values, [2.5, 97.5])
    return float(lo), float(hi)

## 5. Original ALC classifier from the supplied implementation

This block preserves the original mean-based `Cofactor` / `Vitamin` computation used in the supplied `alc.py`.  
It is kept separate from the enhanced classifier so the reviewer can see what comes from the **architecture** and what comes from the **optimizer**.

In [ ]:
class OriginalLiver:
    def __init__(self, num_input, num_output, lobules, seed=42):
        rng = np.random.default_rng(seed)
        self.num_input = num_input
        self.num_output = num_output
        self.lobules = lobules
        self.Cofactor = rng.normal(size=(num_input, lobules)) * np.sqrt(2.0 / num_input)
        self.Vitamin = rng.normal(size=(lobules, num_output)) * np.sqrt(2.0 / num_input)

    def set_weights(self, solution):
        size_cof = self.Cofactor.size
        self.Cofactor = solution[:size_cof].reshape(self.Cofactor.shape)
        self.Vitamin = solution[size_cof:].reshape(self.Vitamin.shape)

    def get_weights(self):
        return np.concatenate([self.Cofactor.ravel(), self.Vitamin.ravel()])

    def reaction(self, toxins):
        A = np.zeros((toxins.shape[0], self.lobules))
        for i in range(self.lobules):
            A[:, i] = (
                np.mean(toxins * self.Cofactor[:, i], axis=1)
                + np.mean(self.Cofactor)
            )

        A = np.maximum(0.0, A)

        B = np.zeros((toxins.shape[0], self.num_output))
        for i in range(self.num_output):
            B[:, i] = (
                np.mean(A * self.Vitamin[:, i], axis=1)
                + np.mean(self.Vitamin)
            )

        B = B - np.max(B, axis=1, keepdims=True)
        exp_b = np.exp(B)
        return exp_b / np.sum(exp_b, axis=1, keepdims=True)

## 6. Original IFOX-style optimizer baseline

The supplied `ifox.py` uses training cross-entropy only.  
The implementation below preserves the same update logic while adding an explicit RNG seed and optimization history for reproducibility.

The seed addition changes reproducibility, not the conceptual objective.

In [ ]:
def categorical_ce(y_true_oh, y_prob):
    p = np.clip(y_prob, 1e-12, 1.0 - 1e-12)
    return float(-np.mean(np.sum(y_true_oh * np.log(p), axis=1)))

class IFOXReproducible:
    def __init__(self, epochs, pop_size, dim, lb, ub, loss_fn, seed=42):
        self.epochs = epochs
        self.pop_size = pop_size
        self.dim = dim
        self.lb = lb
        self.ub = ub
        self.loss_fn = loss_fn
        self.rng = np.random.default_rng(seed)

        self.solutions = [
            self.rng.uniform(lb, ub, size=dim) for _ in range(pop_size)
        ]
        self.best_score = np.inf
        self.best_solution = self.solutions[0].copy()
        self.history = []

    def optimize(self):
        for epoch in range(1, self.epochs + 1):
            for sol in self.solutions:
                fitness = self.loss_fn(sol)
                if fitness < self.best_score:
                    self.best_score = fitness
                    self.best_solution = sol.copy()

            self.history.append({
                "iteration": epoch,
                "objective": float(self.best_score),
            })

            min_alpha = 1.0 / (self.epochs * 2.0)
            alpha = min_alpha + (1.0 - min_alpha) * (1.0 - epoch / self.epochs)

            t = np.mean(self.rng.uniform(0, 1, self.best_solution.size)) / 2.0
            dis = 0.5 * self.best_solution
            jump = max(0.5 * 9.81 * t**2, 1e-12)

            new_solutions = []
            for _ in range(self.pop_size):
                beta = self.rng.uniform(-alpha, alpha, size=self.best_solution.size)
                if self.rng.random() < alpha:
                    candidate = self.best_solution + beta * alpha
                else:
                    candidate = dis * (beta * alpha) / jump

                # IMPORTANT:
                # The original supplied code does not clip after updates.
                # We preserve that behavior for the "Original ALC + IFOX" control.
                new_solutions.append(candidate)

            self.solutions = new_solutions

        return self.best_solution.copy()

## 7. Enhanced ALC architecture

This is the enhanced architecture used in the revised methodology:

\[
H=\mathrm{ReLU}(XW_c+b_h), \qquad Z=HW_v+b_o
\]

It differs intentionally from the original mean-based ALC, which is why the ablation keeps both versions.

In [ ]:
class EnhancedLiver:
    def __init__(self, num_input, num_output, lobules, seed=42):
        rng = np.random.default_rng(seed)
        self.Cofactor = rng.normal(size=(num_input, lobules)) * np.sqrt(2.0 / num_input)
        self.Vitamin = rng.normal(size=(lobules, num_output)) * np.sqrt(2.0 / lobules)
        self.Bile = np.zeros((1, lobules))
        self.Enzyme = np.zeros((1, num_output))

    def set_weights(self, weights):
        a = self.Cofactor.size
        b = a + self.Vitamin.size
        c = b + self.Bile.size

        self.Cofactor = weights[:a].reshape(self.Cofactor.shape)
        self.Vitamin = weights[a:b].reshape(self.Vitamin.shape)
        self.Bile = weights[b:c].reshape(self.Bile.shape)
        self.Enzyme = weights[c:].reshape(self.Enzyme.shape)

    def get_weights(self):
        return np.concatenate([
            self.Cofactor.ravel(),
            self.Vitamin.ravel(),
            self.Bile.ravel(),
            self.Enzyme.ravel(),
        ])

    def reaction(self, X):
        H = np.maximum(0.0, X @ self.Cofactor + self.Bile)
        Z = H @ self.Vitamin + self.Enzyme
        Z = Z - np.max(Z, axis=1, keepdims=True)
        exp_z = np.exp(Z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

## 8. ALC-adapted enhanced Foxtsage-inspired search

The previous notebook used an evolutionary direct-weight search with:

- elitism,
- Gaussian local perturbation,
- random-agent blending,
- bounded clipping,
- adaptive exploration/exploitation.

This differs from the published Foxtsage learning-rate optimizer. The class name below makes that distinction explicit.

The objective used for the proposed framework is:

\[
J(	heta)=0.35L_{fit}+0.65L_{val}+\lambda\,\mathrm{mean}(	heta^2)
\]

with class-weighted fitting loss.

In [ ]:
class EnhancedFoxtsageSearch:
    def __init__(
        self,
        iterations,
        pop_size,
        bounds,
        dim,
        loss_fn,
        seed=42,
        elite_frac=0.20,
    ):
        self.iterations = iterations
        self.pop_size = pop_size
        self.lower, self.upper = bounds
        self.loss_fn = loss_fn
        self.elite_frac = elite_frac
        self.rng = np.random.default_rng(seed)

        self.population = [
            self.rng.uniform(self.lower, self.upper, dim)
            for _ in range(pop_size)
        ]
        self.losses = [self.loss_fn(x) for x in self.population]

        best_idx = int(np.argmin(self.losses))
        self.best_sol = self.population[best_idx].copy()
        self.best_loss = float(self.losses[best_idx])
        self.history = []

    def optimize(self):
        dim = len(self.best_sol)
        elite_count = max(2, int(self.pop_size * self.elite_frac))

        for it in range(self.iterations):
            ranked = np.argsort(self.losses)
            elites = [self.population[i].copy() for i in ranked[:elite_count]]

            new_population = elites.copy()
            step_scale = max(
                0.02,
                0.2 * (1.0 - it / max(1, self.iterations - 1))
            )

            while len(new_population) < self.pop_size:
                if self.rng.random() < 0.70:
                    parent = elites[self.rng.integers(0, elite_count)]
                    child = parent + self.rng.normal(size=dim) * step_scale
                else:
                    random_agent = self.rng.uniform(
                        self.lower, self.upper, size=dim
                    )
                    alpha = self.rng.uniform(0.3, 0.8)
                    child = alpha * self.best_sol + (1.0 - alpha) * random_agent

                child = np.clip(child, self.lower, self.upper)
                new_population.append(child)

            self.population = new_population
            self.losses = [self.loss_fn(x) for x in self.population]

            current_idx = int(np.argmin(self.losses))
            current_loss = float(self.losses[current_idx])
            if current_loss < self.best_loss:
                self.best_loss = current_loss
                self.best_sol = self.population[current_idx].copy()

            self.history.append({
                "iteration": it + 1,
                "objective": float(self.best_loss),
                "step_scale": float(step_scale),
            })

        return self.best_sol.copy()

## 9. Unified trainers for the four ablation variants

The minimum reviewer-facing ablation is:

- **A1** Original ALC + IFOX
- **A2** Original ALC + enhanced Foxtsage-inspired search
- **A3** Enhanced ALC + IFOX
- **A4** Enhanced ALC + enhanced Foxtsage-inspired search (**full proposed framework**)

All four use exactly the same fit/validation/test observations and the same scaling.

In [ ]:
def class_weights_from_labels(y):
    counts = np.bincount(y, minlength=2).astype(float)
    return counts.sum() / (len(counts) * np.maximum(counts, 1.0))

def fit_train_only_ifox(
    liver_cls,
    X_fit,
    y_fit,
    lobules,
    seed,
    iterations=IFOX_ITERATIONS,
    pop_size=IFOX_POPULATION,
):
    liver = liver_cls(X_fit.shape[1], 2, lobules, seed=seed)
    y_fit_oh = one_hot(y_fit)

    def loss_fn(weights):
        liver.set_weights(weights)
        return categorical_ce(y_fit_oh, liver.reaction(X_fit))

    opt = IFOXReproducible(
        epochs=iterations,
        pop_size=pop_size,
        dim=liver.get_weights().size,
        lb=-1.0,
        ub=1.0,
        loss_fn=loss_fn,
        seed=seed,
    )
    best = opt.optimize()
    liver.set_weights(best)
    return liver, pd.DataFrame(opt.history)

def fit_validation_aware_foxtsage(
    liver_cls,
    X_fit,
    y_fit,
    X_val,
    y_val,
    lobules_grid,
    seed,
    iterations=FOXT_ITERATIONS,
    pop_size=FOXT_POPULATION,
    l2_penalty=1e-4,
):
    weights_by_class = class_weights_from_labels(y_fit)
    y_fit_oh = one_hot(y_fit)
    y_val_oh = one_hot(y_val)

    best_record = None

    for lobules in lobules_grid:
        liver = liver_cls(X_fit.shape[1], 2, lobules, seed=seed + lobules)

        def loss_fn(theta):
            liver.set_weights(theta)
            p_fit = np.clip(liver.reaction(X_fit), 1e-12, 1 - 1e-12)
            p_val = np.clip(liver.reaction(X_val), 1e-12, 1 - 1e-12)

            sample_w = weights_by_class[y_fit]
            fit_loss = -np.mean(
                np.sum(sample_w[:, None] * y_fit_oh * np.log(p_fit), axis=1)
            )
            val_loss = -np.mean(
                np.sum(y_val_oh * np.log(p_val), axis=1)
            )
            l2 = l2_penalty * np.mean(theta**2)
            return 0.35 * fit_loss + 0.65 * val_loss + l2

        opt = EnhancedFoxtsageSearch(
            iterations=iterations,
            pop_size=pop_size,
            bounds=(-1.5, 1.5),
            dim=liver.get_weights().size,
            loss_fn=loss_fn,
            seed=seed + lobules,
        )
        best_theta = opt.optimize()
        liver.set_weights(best_theta)

        p_val = np.clip(liver.reaction(X_val), 1e-12, 1 - 1e-12)
        val_ce = categorical_ce(y_val_oh, p_val)

        record = {
            "lobules": lobules,
            "validation_ce": val_ce,
            "liver": liver,
            "history": pd.DataFrame(opt.history),
        }

        if best_record is None or val_ce < best_record["validation_ce"]:
            best_record = record

    return (
        best_record["liver"],
        best_record["history"],
        best_record["lobules"],
        best_record["validation_ce"],
    )

## 10. Run the ablation on prespecified seeds

The final test set is evaluated for every prespecified seed, but **no seed is selected using test performance**.

Seed 42 is the predeclared primary run used for the paper's confusion matrix and ROC/PR figures.  
The full seed set quantifies stochastic variability.

In [ ]:
ablation_specs = {
    "A1 Original ALC + IFOX": {
        "arch": OriginalLiver,
        "optimizer": "ifox",
    },
    "A2 Original ALC + Enhanced Foxtsage Search": {
        "arch": OriginalLiver,
        "optimizer": "foxtsage",
    },
    "A3 Enhanced ALC + IFOX": {
        "arch": EnhancedLiver,
        "optimizer": "ifox",
    },
    "A4 Full EHALC + Enhanced Foxtsage Search": {
        "arch": EnhancedLiver,
        "optimizer": "foxtsage",
    },
}

ablation_rows = []
primary_models = {}

for seed in SEEDS:
    for method, spec in ablation_specs.items():
        t0 = time.perf_counter()

        if spec["optimizer"] == "ifox":
            # A single predeclared architecture for IFOX controls.
            lobules = LOBULES_GRID[0]
            model, history = fit_train_only_ifox(
                spec["arch"], X_fit, y_fit, lobules=lobules, seed=seed
            )
            selected_lobules = lobules
            val_ce = categorical_ce(one_hot(y_val), model.reaction(X_val))
        else:
            model, history, selected_lobules, val_ce = fit_validation_aware_foxtsage(
                spec["arch"],
                X_fit,
                y_fit,
                X_val,
                y_val,
                lobules_grid=LOBULES_GRID,
                seed=seed,
            )

        test_prob = model.reaction(X_test)
        test_pred = np.argmax(test_prob, axis=1)
        m = classification_metrics(y_test, test_pred, test_prob[:, 1])

        elapsed = time.perf_counter() - t0
        row = {
            "method": method,
            "seed": seed,
            "selected_lobules": selected_lobules,
            "validation_ce": float(val_ce),
            "seconds": float(elapsed),
            **m,
        }
        ablation_rows.append(row)

        history.to_csv(
            OUTPUT_DIR / f"history_{method[:2]}_seed{seed}.csv",
            index=False,
        )

        if seed == RANDOM_SEED:
            primary_models[method] = model
            save_predictions(
                OUTPUT_DIR / f"predictions_{method[:2]}_seed{seed}.csv",
                test_idx,
                y_test,
                test_prob,
            )

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(OUTPUT_DIR / "ablation_all_runs.csv", index=False)

display(ablation_df)

## 11. Aggregate repeated-seed ablation results

This table supports the reviewers' request for variability estimates instead of relying on fourth-decimal differences from one stochastic run.

In [ ]:
metric_names = [
    "accuracy", "f1_weighted", "roc_auc", "pr_auc",
    "precision_attack", "recall_attack", "f1_attack"
]

agg_parts = []
for method, g in ablation_df.groupby("method"):
    row = {"method": method, "runs": len(g)}
    for m in metric_names:
        row[f"{m}_mean"] = g[m].mean()
        row[f"{m}_std"] = g[m].std(ddof=1) if len(g) > 1 else 0.0
    row["seconds_mean"] = g["seconds"].mean()
    agg_parts.append(row)

ablation_summary = pd.DataFrame(agg_parts)
ablation_summary.to_csv(OUTPUT_DIR / "ablation_summary.csv", index=False)
display(ablation_summary)

## 12. Conventional baseline models

These are **generic ML/DNN baselines**, not claimed to be direct reimplementations of contemporary EVSE-specific systems.

For the paper, report the tuning space and tuning budget.  
The baseline search uses the **fit subset only** with stratified CV; the separate validation and final test subsets remain outside baseline hyperparameter search.

In [ ]:
baseline_spaces = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_SEED)),
        ]),
        {
            "clf__C": np.logspace(-3, 2, 12),
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            class_weight="balanced_subsample",
        ),
        {
            "n_estimators": [200, 300, 500],
            "max_depth": [None, 10, 20, 40],
            "min_samples_split": [2, 5, 10],
            "max_features": ["sqrt", "log2", 0.5],
        },
    ),
    "HistGradientBoosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_SEED),
        {
            "learning_rate": [0.03, 0.05, 0.1],
            "max_iter": [100, 200, 300],
            "max_leaf_nodes": [15, 31, 63],
            "l2_regularization": [0.0, 1e-4, 1e-3, 1e-2],
        },
    ),
    "SVM (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED)),
        ]),
        {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": ["scale", "auto", 1e-3, 1e-2],
            "clf__class_weight": [None, "balanced"],
        },
    ),
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier()),
        ]),
        {
            "clf__n_neighbors": [3, 5, 7, 9, 11],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],
        },
    ),
    "MLP": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                solver="adam",
                early_stopping=True,
                validation_fraction=0.10,
                n_iter_no_change=10,
                random_state=RANDOM_SEED,
            )),
        ]),
        {
            "clf__hidden_layer_sizes": [(128,), (256, 128), (512, 256, 128)],
            "clf__alpha": [1e-5, 1e-4, 1e-3],
            "clf__learning_rate_init": [3e-4, 1e-3, 3e-3],
            "clf__max_iter": [80, 120, 160],
        },
    ),
}

N_ITER_SEARCH = 3 if QUICK_MODE else 12
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

baseline_rows = []
best_baseline_params = {}

for name, (model, params) in baseline_spaces.items():
    search = RandomizedSearchCV(
        model,
        param_distributions=params,
        n_iter=N_ITER_SEARCH,
        scoring="roc_auc",
        cv=inner_cv,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_all[fit_idx], y_fit)

    # Validation is reported, but not used to retune after RandomizedSearchCV.
    val_pred = search.best_estimator_.predict(X_all[val_idx])
    val_score = search.best_estimator_.predict_proba(X_all[val_idx])[:, 1]
    val_metrics = classification_metrics(y_val, val_pred, val_score)

    test_pred_b = search.best_estimator_.predict(X_all[test_idx])
    test_score_b = search.best_estimator_.predict_proba(X_all[test_idx])[:, 1]
    test_metrics = classification_metrics(y_test, test_pred_b, test_score_b)

    baseline_rows.append({
        "method": name,
        "search_best_cv_roc_auc": float(search.best_score_),
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    best_baseline_params[name] = search.best_params_

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(OUTPUT_DIR / "baseline_results.csv", index=False)

with open(OUTPUT_DIR / "baseline_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_baseline_params, f, indent=2, default=str)

display(baseline_df)

## 13. Final primary prediction file and internally consistent figures

The predeclared primary run is:

**A4 Full EHALC + Enhanced Foxtsage Search, seed = 42**

Every headline result below is recomputed from its saved `final_holdout_predictions.csv`.  
This prevents the Abstract, Table 1, ROC/PR figure and confusion matrix from drifting apart.

In [ ]:
primary_method = "A4 Full EHALC + Enhanced Foxtsage Search"
primary_model = primary_models[primary_method]

final_prob = primary_model.reaction(X_test)
final_predictions = save_predictions(
    OUTPUT_DIR / "final_holdout_predictions.csv",
    test_idx,
    y_test,
    final_prob,
)

final_metrics, final_cm = evaluate_prediction_file(
    OUTPUT_DIR / "final_holdout_predictions.csv"
)

with open(OUTPUT_DIR / "final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2)

pd.DataFrame(
    final_cm,
    index=["Actual Benign", "Actual Attack"],
    columns=["Predicted Benign", "Predicted Attack"],
).to_csv(OUTPUT_DIR / "final_confusion_matrix.csv")

print(json.dumps(final_metrics, indent=2))
print("\nConfusion matrix:\n", final_cm)

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(final_cm)
ax.set_xticks([0, 1], ["Benign", "Attack"])
ax.set_yticks([0, 1], ["Benign", "Attack"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Final Holdout Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(final_cm[i, j]), ha="center", va="center")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "final_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# ROC
y_score = final_predictions["prob_attack"].to_numpy()
fpr, tpr, _ = roc_curve(y_test, y_score)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {final_metrics['roc_auc']:.6f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Final Holdout ROC Curve")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "final_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# PR
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(recall_curve, precision_curve, label=f"PR-AUC = {final_metrics['pr_auc']:.6f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Final Holdout Precision–Recall Curve")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "final_pr_curve.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. Bootstrap 95% confidence intervals for the primary run

These intervals characterize sampling uncertainty on the fixed holdout set.

In [ ]:
yp = final_predictions["y_pred"].to_numpy()
ys = final_predictions["prob_attack"].to_numpy()

ci_rows = []
for metric in ["accuracy", "f1_weighted", "roc_auc", "pr_auc"]:
    lo, hi = bootstrap_ci(y_test, yp, ys, metric=metric, n_boot=500 if QUICK_MODE else 5000)
    ci_rows.append({
        "metric": metric,
        "estimate": final_metrics[metric],
        "ci95_low": lo,
        "ci95_high": hi,
    })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(OUTPUT_DIR / "final_bootstrap_ci.csv", index=False)
display(ci_df)

## 15. Global SHAP analysis

The previous notebook explained only `X_test[:1]`.  
This section aggregates attribution over multiple holdout samples and ranks features using mean absolute SHAP magnitude.

Because the enhanced ALC is a custom NumPy model with 905+ inputs, model-agnostic SHAP can be computationally expensive. The sample sizes are therefore explicitly reported and configurable.

In [ ]:
if RUN_SHAP:
    import shap

    rng = np.random.default_rng(RANDOM_SEED)

    bg_n = min(SHAP_BACKGROUND_N, len(X_fit))
    explain_n = min(SHAP_EXPLAIN_N, len(X_test))

    bg_idx = rng.choice(len(X_fit), size=bg_n, replace=False)
    explain_idx = rng.choice(len(X_test), size=explain_n, replace=False)

    background = X_fit[bg_idx]
    X_explain = X_test[explain_idx]

    def attack_probability(z):
        return primary_model.reaction(np.asarray(z))[:, 1]

    masker = shap.maskers.Independent(background)
    explainer = shap.Explainer(
        attack_probability,
        masker,
        algorithm="permutation",
        feature_names=feature_names,
    )

    shap_values = explainer(
        X_explain,
        max_evals=2 * X_explain.shape[1] + 1,
    )

    mean_abs = np.abs(shap_values.values).mean(axis=0)
    shap_global = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs,
    }).sort_values("mean_abs_shap", ascending=False)

    shap_global.to_csv(OUTPUT_DIR / "global_shap_importance.csv", index=False)
    display(shap_global.head(20))

    top = shap_global.head(20).sort_values("mean_abs_shap")
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(top["feature"], top["mean_abs_shap"])
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(f"Global Feature Attribution (n={explain_n})")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "global_shap_top20.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("SHAP skipped. Set RUN_SHAP=True for the paper run.")

## 16. Optional cross-scenario stress test

The source dataset includes `Scenario` metadata (e.g., Cryptojacking, Recon, DoS).  
This metadata is not used as a predictor.

A leave-one-attack-scenario-out experiment can provide stronger evidence about generalization.  
This section prepares the splits only; it does not fabricate results.

In [ ]:
if "Scenario" in metadata.columns:
    scenario_series = metadata["Scenario"].astype(str).str.strip()
    print(scenario_series.value_counts().head(20))

    attack_scenarios = [
        s for s in scenario_series[y_all == 1].dropna().unique()
        if s.lower() not in {"nan", "0", "benign"}
    ]
    print("Attack scenarios available for leave-one-scenario-out:", attack_scenarios)
else:
    attack_scenarios = []
    print("Scenario metadata not available.")

## 17. Reproducibility manifest

This file records the exact experimental configuration used for the run.

In [ ]:
manifest = {
    "random_seed_primary": RANDOM_SEED,
    "seeds": SEEDS,
    "n_valid_samples": int(len(y_all)),
    "n_features": int(X_all.shape[1]),
    "split_sizes": {
        "fit": int(len(fit_idx)),
        "validation": int(len(val_idx)),
        "test": int(len(test_idx)),
    },
    "ifox": {
        "iterations": IFOX_ITERATIONS,
        "population": IFOX_POPULATION,
        "bounds_initialization": [-1.0, 1.0],
        "post_update_clipping": False,
    },
    "enhanced_foxtsage_search": {
        "iterations": FOXT_ITERATIONS,
        "population": FOXT_POPULATION,
        "lobules_grid": list(LOBULES_GRID),
        "bounds": [-1.5, 1.5],
        "elite_fraction": 0.20,
        "local_exploitation_probability": 0.70,
        "exploration_probability": 0.30,
        "blend_alpha_range": [0.3, 0.8],
        "objective_alpha_fit": 0.35,
        "objective_beta_validation": 0.65,
        "l2_penalty": 1e-4,
    },
    "primary_model": primary_method,
    "primary_seed": RANDOM_SEED,
    "quick_mode": QUICK_MODE,
}

with open(OUTPUT_DIR / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

# Reviewer mapping

### Reviewer 1 — Comments 1–2
Use:
- `final_holdout_predictions.csv`
- `final_metrics.json`
- `final_confusion_matrix.csv`
- ROC and PR figures generated directly from the same prediction file.

### Reviewer 1 — Comments 3–4
Use:
- `split_indices.npz`
- `split_summary.csv`
- `experiment_manifest.json`
- optimization history CSV files.

### Reviewer 1 — Comment 5 / Reviewer 2 — Comment 2
Use:
- `ablation_all_runs.csv`
- `ablation_summary.csv`.

### Reviewer 1 — Comment 6
Use:
- `baseline_best_params.json`
- `baseline_results.csv`
- the explicit tuning spaces in this notebook.

### Reviewer 1 — Comment 8
Use:
- `global_shap_importance.csv`
- `global_shap_top20.png`.

### Reviewer 1 — Comment 9 / Reviewer 2 — Comment 6
Release this notebook and the contents of `revision_outputs/` together with environment requirements.

### Reviewer 2 — Comment 4
Use the optional cross-scenario section to implement a leave-one-attack-scenario-out stress test if computationally feasible.

### Reviewer 2 — Comment 5
Use repeated-seed mean ± standard deviation plus the bootstrap 95% confidence intervals.

## Before manuscript resubmission

Do **not** copy the previous manuscript's 0.998819 / 0.999931 / 0.999920 values into the revision unless this notebook reproduces them from `final_holdout_predictions.csv`.

The revised manuscript should use only the newly generated values.